# Reel Brain Report

See how an average brain is **predicted** to respond to your reel, second by second, using Meta FAIR's
[TRIBE v2](https://github.com/facebookresearch/tribev2). This notebook runs everything on Google Colab's
free GPU and gives you a `report.zip` to open in the report viewer.

**Before you start (one time only)**
1. A [Hugging Face](https://huggingface.co) account. Request access to
   [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) and wait for the approval email
   (minutes to a few hours).
2. A **Read** token from https://huggingface.co/settings/tokens. In Colab, click the 🔑 key icon on the left →
   **Add new secret** → name `HF_TOKEN`, paste the token as the value, and switch on **Notebook access**.

**Each time**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. Run the cells **one at a time, top to bottom** (click ▶ and wait for each to finish).
   Step 2 restarts Colab on purpose; just carry on with step 3 afterwards.

Reels must be **60 seconds or shorter**. Expect roughly 15–40 minutes on the free T4, depending on length.

*These are model predictions of an average brain's response, not measurements of you or your audience.
TRIBE v2 is licensed CC BY-NC 4.0 (non-commercial use only); Llama 3.2 is under the Llama 3.2 Community
License. By running this you agree to both.*

## 1. Check the GPU
You should see a **Tesla T4** (or another NVIDIA GPU). If you get an error, set the runtime to T4 GPU (see above).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install (about 5 minutes)
When it finishes, Colab restarts by itself and may say the session crashed. That's expected. Continue with step 3.

In [ ]:
![ -d /content/tribev2 ] || git clone -q https://github.com/facebookresearch/tribev2 /content/tribev2
!cd /content/tribev2 && git checkout -q af58661791a351a448a489042a28f6c37e1c14b7 && pip install -q -e .
# tribev2 installs torch 2.6.0; Colab's preinstalled torchvision/torchaudio target a newer torch.
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
# The report builder from this project (no-deps so it can't change TRIBE's versions).
!pip install -q nilearn
!pip install -q --no-deps "git+https://github.com/ammarahmad1/reel-brain-analysis.git#subdirectory=packages/report"
# whisperx needs NLTK's punkt_tab; NLTK refuses to fetch it through Colab's proxy, so fetch it directly.
!mkdir -p /root/nltk_data/tokenizers && wget -q https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/tokenizers/punkt_tab.zip -O /tmp/punkt_tab.zip && unzip -q -o /tmp/punkt_tab.zip -d /root/nltk_data/tokenizers/

print("Installed. Restarting Colab so the new versions load...")
import os
os.kill(os.getpid(), 9)

## 3. Sign in to Hugging Face
Uses the `HF_TOKEN` secret you added (see the top of this notebook).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    raise SystemExit(
        "No HF_TOKEN found. Click the key icon on the left, add a secret named HF_TOKEN with your "
        "Hugging Face token, switch on Notebook access, then run this cell again."
    )
login(token=token)
print("Signed in.")

## 4. Choose your reel
Pick one video (MP4, MOV or WebM, **60 seconds or shorter**) that you made or have the rights to use.
Already added it in Colab's file panel? Paste its path into `EXISTING_VIDEO` instead of uploading again.

In [ ]:
import shutil, subprocess
from pathlib import Path
from google.colab import files

EXISTING_VIDEO = ""  # e.g. "/content/my_reel.mp4"; leave empty to get an upload button
MAX_SECONDS = 60
VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi"}

reel_dir = Path("/content/reel")
shutil.rmtree(reel_dir, ignore_errors=True)
reel_dir.mkdir()
if EXISTING_VIDEO:
    shutil.copy(EXISTING_VIDEO, reel_dir)
else:
    for name, data in files.upload().items():
        (reel_dir / Path(name).name).write_bytes(data)
        Path(name).unlink(missing_ok=True)

videos = [p for p in reel_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS]
if len(videos) != 1:
    raise SystemExit("Please choose exactly one video file (MP4, MOV or WebM).")
duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(videos[0])],
    capture_output=True, text=True, check=True).stdout)
if duration > MAX_SECONDS + 0.5:
    raise SystemExit(f"This reel is {duration:.0f}s long. Please trim it to {MAX_SECONDS} seconds or less.")
print(f"Ready: {videos[0].name} ({duration:.1f}s)")

## 5. Run TRIBE v2 and build the report
The first run downloads about 15 GB of model weights, then analyses your reel. This is the long step:
keep this tab open. You'll see warnings scroll past; they're normal.

In [ ]:
import json, shutil, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, "/content/tribev2")  # otherwise `import tribev2` picks up the repo folder itself
import numpy as np
from tribev2 import TribeModel
from reel_report import build_report

VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi"}
video = next(p for p in Path("/content/reel").iterdir() if p.suffix.lower() in VIDEO_EXTS)
out = Path("/content/report")
shutil.rmtree(out, ignore_errors=True)
out.mkdir()

start = time.time()
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="/content/cache")
events = model.get_events_dataframe(video_path=str(video))
preds, segments = model.predict(events=events)

duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(video)],
    capture_output=True, text=True, check=True).stdout)
report = build_report(
    np.asarray(preds),
    [float(s.start) for s in segments],
    duration,
    word_count=int((events["type"] == "Word").sum()),
    heatmap_dir=out,
)
(out / "report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

ins = report["insights"]
print(f"\nDone in {(time.time() - start) / 60:.0f} min.\n")
print(f"Hook (first {ins['hook']['end_s']}s): {ins['hook']['rating']}")
print(f"Peak moment: {ins['peak']['second']}s")
print(f"Carried by: {ins['carrier']['channel'] or 'no single channel'}")
print("\nTop fixes:")
for i, fix in enumerate(report["fixes"], 1):
    print(f"  {i}. {fix['title']}")

## 6. Download your report
Downloads `report.zip`. Open the report viewer and drop the zip onto it to see the full report.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/report", "zip", "/content/report")
files.download("/content/report.zip")

---
Optional: delete your reel and the extracted audio from this Colab session (Colab also wipes everything when the
session ends).

In [ ]:
import shutil
shutil.rmtree("/content/reel", ignore_errors=True)
print("Deleted.")